# Week 8: Řešení soustav nelineárních algebraických rovnic

**Assignment:** [![Run in JupyterHub](https://img.shields.io/badge/Run%20in-JupyterHub-F37626?logo=jupyter)](https://hpc.troja.mff.cuni.cz:8000/hub/user-redirect/git-pull?app=lab&repo=https%3A%2F%2Fgitlab.karlin.mff.cuni.cz%2Fznm%2Fcviceni-student&branch=main&subpath=notebooks%2Fassignment%2Fweek-08-root-finding.ipynb) [![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-blue)](https://gitlab.karlin.mff.cuni.cz/znm/cviceni-student/-/raw/main/notebooks/assignment/week-08-root-finding.ipynb?inline=false) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/num-math/cviceni-student/blob/main/notebooks/assignment/week-08-root-finding.ipynb)

**Solution:** [![Run in JupyterHub](https://img.shields.io/badge/Run%20in-JupyterHub-F37626?logo=jupyter)](https://hpc.troja.mff.cuni.cz:8000/hub/user-redirect/git-pull?app=lab&repo=https%3A%2F%2Fgitlab.karlin.mff.cuni.cz%2Fznm%2Fcviceni-student&branch=main&subpath=notebooks%2Fsolution%2Fweek-08-root-finding.ipynb) [![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-blue)](https://gitlab.karlin.mff.cuni.cz/znm/cviceni-student/-/raw/main/notebooks/solution/week-08-root-finding.ipynb?inline=false) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/num-math/cviceni-student/blob/main/notebooks/solution/week-08-root-finding.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Materiály

- [poznámky z přednášky - algebraické rovnice &  Newtonova metoda](https://www.karlin.mff.cuni.cz/~kucera/ZNM/1%20-%20Newtonova%20metoda.pdf)
- [poznámky z přednášky - iterace funkcí & věta o pevném bodu](https://www.karlin.mff.cuni.cz/~kucera/ZNM/2%20-%20Iterace%20funkci.pdf)
- další materiály
  - skripta [Kučera: Numerical methods for nonlinear equations](https://www.karlin.mff.cuni.cz/%7Ekucera/Numerical_Methods_for_Nonlinear_Equations.pdf) (Sekce 2.1, 2.3, 3.1, 3.2, 3.3)
  - skripta [Kelley: Solving Nonlinear Equations with Newton's Method](https://cuni.primo.exlibrisgroup.com/discovery/fulldisplay?docid=alma990013579040106986&context=L&vid=420CKIS_INST:UKAZ&lang=cs&search_scope=MyInst_and_CI&adaptor=Local%20Search%20Engine&tab=Everything&query=any,contains,Solving%20Nonlinear%20Equations%20with%20Newton%27s%20Method&mode=basic) (knihovna v karlíně)
  - skripta [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (Volume 1, Kapitola 10)

- ***Primární úlohy*** : A1, B1, B2

- ***Úlohy k bonusové části zkoušky*** : A3, B2, B3

- základní ***Learning outcomes*** (cviko, nikoliv přednáška)
  - Umím graficky demonstrovat různé vývoje aproximací Newtonovy metody (konvergence/divergence/oscilace) pro skalární problém.
  - Dokážu odvodit Newtonovu metody pro hledání kořenů vektorových funkcí.
  - Dokážu graficky znázornit/rozpoznat rozdíl mezi lineární a kvadratickou konvergencí posloupnosti.
  - Dokážu odvodit a analyzovat jednoduché fixed-point metody pro konkrétní reálné funkce.
  - Umím graficky demonstrovat různé vývoje aproximací fixed-point metody (konvergence/divergence/oscilace) pro skalární problém.

## Intro

V praxi je řešení soustav rovnic jednou z nejdůležitějších, ne-li tou nejdůležitější, operací a přirozeně nezůstává pouze u těch lineárních, které jsme řešili doposud. Mezi důležité aplikace patří například řešení diferenciálních rovnic nebo optimalizace.

Náš problém je tedy zadán jako

<a name="eq1"></a>
\begin{equation*}
\tag{1}
\text{Pro danou funkci $\mathbf{F} \, : \, \mathbb{R}^{n} \rightarrow \mathbb{R}^{n}$ najdi $\mathbf{x}^{*}$ tak aby $\mathbf{F}(\mathbf{x}^{*})=0$.}
\end{equation*}

V principu víme, jak tento problém „přesně“ řešit, pouze pokud je $\mathbf{F}$ ***afinní***, tj. $\mathbf{F} , : , \mathbf{x} \mapsto A\mathbf{x} - \mathbf{b}$ pro danou matici $A$ a vektor $\mathbf{b}$ vhodných rozměrů. Pro takovou funkci hledání kořenů odpovídá řešení $A\mathbf{x} = \mathbf{b}$. Zde máme vybudovanou teorii z lineární algebry a různé postupy pro řešení (Gaussova eliminace, QR-rozklad, ...).

Ale aplikace jsou bohužel neúprosné a často kořeny nelineárních funkcí zkrátka potřebujeme.

Bez explicitního vzorce řešení, který bychom mohli numericky aproximovat, přicházejí na řadu ***iterační metody***, tj. metody, které na základě aproximace $\mathbf{x}_0 \approx \mathbf{x}^*$ spočítají aproximaci (snad) lepší. To si můžeme napsat jako $\Phi(\mathbf{x}_0) =: \mathbf{x}_1 \approx \mathbf{x}^*$.

Idea je vcelku přímočará – vezmeme si nějaký počáteční odhad $\mathbf{x}_0$ a pomocí iterační metody ho zpřesňujeme, a tím získáváme posloupnost aproximací

\begin{equation*}
\mathbf{x}_0, \; \mathbf{x}_1 = \Phi(\mathbf{x}_0), \; \mathbf{x}_2 = \Phi(\mathbf{x}_1) = \Phi( \Phi(\mathbf{x}_0) ), \dotsc ,
\end{equation*}

kde obecný $k$-tý prvek je dán jako $\Phi^k(\mathbf{x}_0) = \underbrace{\Phi(\dotsc (\Phi(\mathbf{x}_0)) )}_{k\text{-krat}}$. Tomuto „vrstvení“ aplikace funkce se říká [iterated function](https://en.wikipedia.org/wiki/Iterated_function#Fractional_iterates_and_flows,_and_negative_iterates).

Na přednášce jsme viděli dvě základní myšlenky.

[***Metody typu Newton***](https://en.wikipedia.org/wiki/Newton%27s_method)

 >Aproximujeme naši funkci $\mathbf{F}$ v bodě $\mathbf{x}_0$ lineární funkcí $\mathbf{L}_{\mathbf{x}_0}$ a místo $\mathbf{F}(\mathbf{x}) = 0$ budeme řešit $\mathbf{L}_{\mathbf{x}_0}(\mathbf{x}) = 0$. Řešení $\mathbf{L}_{\mathbf{x}_0}(\mathbf{x}) = 0$ označíme jako naše $\mathbf{x}_1$, a tedy dostáváme

>
>\begin{equation*}
\Phi \, : \, \mathbf{x}_0 \mapsto \mathbf{x}_1, \quad \text{kde $\mathbf{x}_1$ bereme tak, aby $\mathbf{L}_{\mathbf{x}_0}(\mathbf{x}_1) = 0$.}
\end{equation*}
>
> Asi nejpřirozenějším způsobem jak volit $\mathbf{L}_{\mathbf{x}_0}$ na základě $\mathbf{F}$ je aplikace Taylorova rozvoje. Označíme-li matici prvních derivací funkce $\mathbf{F}$ v bodě $\mathbf{x}_0$ jako $J_{\mathbf{x}_0}$, pak máme pro $\mathbf{x}$ v okolí $\mathbf{x}_0$
>
>\begin{equation*}
\mathbf{F}(\mathbf{x}) \approx \mathbf{F}(\mathbf{x}_0) + J_{\mathbf{x}_0}(\mathbf{x}-\mathbf{x}_0) + \mathcal{O} \left( \| \mathbf{x} - \mathbf{x}_0 \|^2 \right)
\end{equation*}
>
> a tedy dostáváme
>
>\begin{equation*}
\mathbf{L}_{\mathbf{x}_0} \, : \, \mathbf{x} \mapsto  \mathbf{F}(\mathbf{x}_0) + J_{\mathbf{x}_0}(\mathbf{x}-\mathbf{x}_0), \quad \text{a tedy} \\
\Phi \, : \, \mathbf{x}_0 \mapsto \mathbf{x}_1, \quad \text{kde $\mathbf{x}_1$ bereme tak, aby $J_{\mathbf{x}_0}\mathbf{x}_1 = J_{\mathbf{x}_0}\mathbf{x}_0 - \mathbf{F}(\mathbf{x}_0)$.}
\end{equation*}
>
> Pokud budeme s $\mathbf{x}_0$ dostatečně blízko kořenu $\mathbf{x}^*$ (a nebo pokud se dostatečně blízko dostaneme v naší posloupnosti $\mathbf{x}_0,\mathbf{x}_1,\dotsc$) pak Newtonova metoda bude konvergovat alespoň kvadraticky.
> Způsobů, jak vybrat $\mathbf{L}_{\mathbf{x}_0}$ je více - obecně takové metody nazíváme tzv. *quasi-Newton methods*.
>
> Všimněme si, že každá iterace těchto metod vyžaduje řešení soustavy *lineárních* algebraických rovnic - jinými slovy, převádíme nelineární problém na sekvenci lineárních, tzv. "linearizujeme".

[***Fixed-point metody***](https://en.wikipedia.org/wiki/Fixed-point_iteration)
> Obecně tyto metody řeší tzv. *fixed point problém*, tj.
>
>\begin{equation*}
\text{Pro danou funkci $\mathbf{G} \, : \, \mathbb{R}^{n} \rightarrow \mathbb{R}^{n}$ najdi $\mathbf{x}^{*}$ tak aby $\mathbf{G}(\mathbf{x}^{*})=\mathbf{x}^{*}$.}
\end{equation*}
>
> Idea je volit $\mathbf{G}$ *chytře*, tj. tak aby platilo
>
>\begin{equation*}
 \mathbf{G}(\mathbf{x}^{*})=\mathbf{x}^{*} \quad \Rightarrow \quad \mathbf{F}(\mathbf{x}^{*}) = 0
\end{equation*}
>
> a my mohli použít metody na hledání pevného bodu $\mathbf{G}$ k nalezení kořene $\mathbf{F}$.
> Na hledání pevného bodu dané funkce $\mathbf{G}$ máme z přednášky jeden nástroj a to je [Banachova věta o kontrakci](https://en.wikipedia.org/wiki/Banach_fixed-point_theorem#Applications):
  - Mějme funkci $\mathbf{G}\, :\, \mathcal{M}\rightarrow \mathcal{M}$, pro kterou platí
    - $\mathbf{G} : \mathcal{M}\rightarrow \mathcal{M}$ je spojitá,
    - $\mathcal{M} \subset \mathbb{R}^{n}$ je uzavřená,
    - $\mathbf{G}(\mathbf{x})$ je kontrakce.
>
> - Pak volbou posloupnost $\mathbf{x}_0,\mathbf{G}(\mathbf{x}_0),\mathbf{G}(\mathbf{G}(\mathbf{x}_0)), \dotsc$ konverguje k pevnému bodu $\mathbf{x}^{*}$ funkce $\mathbf{G}$. Jinými slovy, bereme $\Phi = \mathbf{G}$ a počítáme posloupnost aproximací jako výše.
>
> Díky Banachově větě o kontrakci máme zaručenou alespoň lineární konvergenci této posloupnosti k pevnému bodu $\mathbf{G}$ a tedy i ke kořenu $\mathbf{F}$.

### Úloha 0

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

Co lze/nelze říci o podmíněnosti problému [(1)](#eq1)? Uveďte příklad, kdy je dobře podmíněný a kdy je špatně podmíněný.

</blockquote>
</div>

## Část A: Newtonova metoda

### Shrnutí z přednášky

Jak jsme viděli výše, tak pro použití Newtonovy metody pro nalezení kořene funkce $\mathbf{F}$ budeme opakovaně používat předpis

\begin{equation*}
J_{\mathbf{x}_k}\mathbf{x}_{k+1} = J_{\mathbf{x}_k}\mathbf{x}_k - \mathbf{F}(\mathbf{x}_k) \quad \text{pro $k=1,2,\dotsc$,}
\end{equation*}

který odpovídá řešení lineárního systému

\begin{equation*}
\mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{v}_k, \quad \text{kde} \quad J_{\mathbf{x}_k} \mathbf{v} = - \mathbf{F}(\mathbf{x}_k) \quad \text{pro $k=1,2,\dotsc$,}
\end{equation*}


Je tedy zjevné, že budeme potřebovat vyhodnocovat nejen $\mathbf{F}$ ale i všechny parciální derivace $\mathbf{F}$, tj. tzv. Jakobián

\begin{equation*}
J_{\mathbf{x}_k} :=
\begin{bmatrix}
\frac{\partial F_1 }{ \partial x_1 } & \frac{ \partial F_1 }{ \partial x_2 } & \dots & \frac{ \partial F_1 }{ \partial x_n } \\
\frac{ \partial F_2 }{ \partial x_1 } & \frac{ \partial F_2 }{ \partial x_2 } & \dots & \frac{ \partial F_2 }{ \partial x_n } \\
\vdots & \vdots & \ddots & \vdots \\
\frac{ \partial F_n }{ \partial x_1 } & \frac{ \partial F_n }{ \partial x_2 } & \dots & \frac{ \partial F_n }{ \partial x_n } \\
\end{bmatrix}(\mathbf{x}_k)
\equiv
\begin{bmatrix}
\frac{\partial F_1 }{ \partial x_1 }(\mathbf{x}_k) & \frac{ \partial F_1 }{ \partial x_2 }(\mathbf{x}_k) & \dots & \frac{ \partial F_1 }{ \partial x_n }(\mathbf{x}_k) \\
\frac{ \partial F_2 }{ \partial x_1 }(\mathbf{x}_k) & \frac{ \partial F_2 }{ \partial x_2 }(\mathbf{x}_k) & \dots & \frac{ \partial F_2 }{ \partial x_n }(\mathbf{x}_k) \\
\vdots & \vdots & \ddots & \vdots \\
\frac{ \partial F_n }{ \partial x_1 }(\mathbf{x}_k) & \frac{ \partial F_n }{ \partial x_2 }(\mathbf{x}_k) & \dots & \frac{ \partial F_n }{ \partial x_n }(\mathbf{x}_k) \\
\end{bmatrix}.
\end{equation*}

My se budeme zajímat o chování Newtonovy metody.

### Úloha A1 - implementace

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Naimplementujte funkci `newton_method()` podle anotace níže.
  - Naimplementujte metodu za použití funkce [`np.linalg.solve()`](https://numpy.org/doc/2.2/reference/generated/numpy.linalg.solve.html) (tj., nepoužívejte funkci `np.linalg.inv()`),
  - Dávejte si pozor na správnou implementaci konvergenčního kritéria. Možná se bude hodit příkaz [`break`](https://www.geeksforgeeks.org/python/python-break-statement/).

</blockquote>
</div>

In [ ]:
def newton_method(f, df, x0, max_iter, tol):
    """
    Newtonova metoda pro reseni soustavy nelinearnich algebraickych rovnic "f(x)=0".

    Input
    -------
    f :   python funkce (callable). Funkce vyhodnocujici hodnoty "f : R^n -> R^n",
          syntax "f( np.array([x1,...,xn]) )". Tj. nevolame funkci jako funkci "n" promennych, ale jako funkci s jednim inputem - vektorem z R^n
    df :  python funkce (callable). Funkce vyhodnocujici hodnoty derivace funkce "f", "df : R^n -> R^{n*n}"
          syntax "f( np.array([x1,...,xn]) )". Tj. nevolame funkci jako funkci "n" promennych, ale jako funkci s jednim inputem - vektorem z R^n
    x0 :  np.array dimenze "n" s prvky np.float64. Pocatecni aproximace korene
    max_iter : np.int32. Maximalni pocet iteraci algoritmu, max_iter > 0
    tol : np.float64. Tolerance chyby ve vyslednem reseni - vyzadujeme, aby zaroven platilo:
            - "norm( x_k - x_{k-1} ) < tol",
            - "norm( f(x_k) ) < tol",
          tzn., algoritmus vrati output po iteraci "k", ve ktere jsou obe tyto podminky splneny.
          Pokud pro nejake "k < max_iter" jsou obe podminky splneny, rekneme, ze algoritmus "zkonvergoval" a uz nebudeme pocitat dalsi aproximace.
          Pokud pro zadne  "k < max_iter" nejsou obe podminky splneny, spocitame "k" iteraci.

    Output
    -------
    x_approx: np.array dimenze "d-krat-(max_iter+1)". Musi platit
    "x_approx[:,i] = xi", kde "xi" je i-ta iterace/aproximace korene "f" (tj. na vystupu vracime celou posloupnost aproximaci, vcetne te nulte)
    n_iter:    np.int32. Pocet provedenych iteraci

    Poznamka
    -------
    Aproximace x0, x1, ... generovane timto algoritmem by mely byt ulozeny v poli
    "x_approx[0:n_iter+1,:]" - radek po radku. Konkretne finalni aproximace bude ulozena v poli "x_approx[:,n_iter]".
    """
    x_approx = np.nan * np.zeros((np.size(x0), max_iter + 1))
    x_approx[:,0] = x0

    for k in range(1, max_iter + 1):
        ...

    return x_approx, k

####################################################################################
def test_newton_method():
    def f_square(x):
        return x**2
    def df_square(x):
        return 2*x*np.eye(len(x))

    def f_cos_sq(x):
        return np.cos(x**2)
    def df_cos_sq(x):
        return -2*x*np.sin(x**2)*np.eye(len(x))     # pro newton_method (pole -> matice)
    def df_cos_sq_scalar(x):
        return -2*x*np.sin(x**2)                    # pro scipy.optimize.newton (skalar -> skalar)

    x0 = 5; maxit = 10; tol = 1e-8
    x_approx, n_iter = newton_method(f_square, df_square, x0, maxit, tol)
    assert x_approx[:,0] == x0, "Neplati 'x_approx[:,0] = x0' pro skalarni funkce."
    assert x_approx.shape == (np.size(x0),maxit+1), "Ouput 'x_approx' ma rozmery {}, pro skalarni funkci a 'max_iter = {}'.".format(x_approx.shape,maxit)
    assert np.allclose( x_approx.reshape((np.size(x_approx),)), 5*(2.)**(np.arange(0,-(maxit+1),-1)) ), "Spatny output pro skalarni pripad 'f(x)=x^2, x0={}, max_iter={}, tol={}' - porovnejte s vypoctem tuzka-a-papir.".format(x0,maxit,tol)

    x0 = 5*np.ones(5); maxit = 10; tol = 1e-8
    x_approx, n_iter = newton_method(f_square, df_square, x0, maxit, tol)
    assert np.linalg.norm(x_approx[:,0] - x0) < 1e-15, "Neplati 'x_approx[:,0] = x0' pro vektorove funkce."
    assert x_approx.shape == (np.size(x0),maxit+1),"Ouput 'x_approx' ma rozmery {}, pro vektorovou funkci z R{} do R{} a 'max_iter = {}'.".format(x_approx.shape,np.size(x0),np.size(x0),maxit)
    assert np.allclose( x_approx, np.outer( np.ones(np.size(x0)) , 5*(2.)**(np.arange(0,-(maxit+1),-1))  ) ), "Spatny output pro skalarni pripad 'f(x)=[x^2,...,x^2], x0={}, max_iter={}, tol={}' - porovnejte s vypoctem tuzka-a-papir.".format(x0,maxit,tol)

    x0 = 1; maxit = 10; tol = 1e-6
    x_approx, n_iter = newton_method(f_cos_sq, df_cos_sq, x0, maxit, tol)
    newton_scipy = newton(f_cos_sq, x0, fprime=df_cos_sq_scalar, maxiter=maxit, tol=tol)
    assert np.abs( x_approx[:,n_iter] - newton_scipy) <= 1e-12, "Spatny output funkce newton_method() pro 'f(x)=cos(x^2), x0={}, max_iter={}, tol={}'. \n Vysledna aproximace ma byt {}".format(x0,maxit,tol,newton_scipy)

    x0 = 1; maxit = 10; tol = 1e-14
    x_approx, n_iter = newton_method(f_cos_sq, df_cos_sq, x0, maxit, tol)
    newton_scipy = newton(f_cos_sq, x0, fprime=df_cos_sq_scalar, maxiter=maxit, tol=tol)
    assert x_approx.shape == (np.size(x0),maxit+1), "Spatny output funkce Newton()"
    assert np.abs( x_approx[:,n_iter] - newton_scipy) <= 1e-12, "Spatny output funkce newton_method() pro 'f(x)=cos(x^2), x0={}, max_iter={}, tol={}'. \n Vysledna aproximace ma byt {}".format(x0,maxit,tol,newton_scipy)
    return '"newton_method()" je OK'

from scipy.optimize import newton
test_newton_method()

### Úloha A2 - Newtonova metoda a podmíněnost & stabilita

V A1 jsme Newtonovu metodu naimplementovali jako a tedy se můžeme přirozeně ptát, naše implementace odpovídá definici *stabilního algoritmu* ze začátku semestru. To je *velmi* obtížná otázka. Druhá otázka, která se nabízí, je následující:

> Je "matematický problém" $\mathcal{P} : \mathbf{x}_0 \mapsto \mathbf{x}_k$, kde $\mathbf{x}_k$ odpovídá $k$-té iteraci Newtonovi metody, dobře podmíněný?
> ***Poznámka***: místo slova "podmíněnost" často používáme slovo "citlivost", například říkáme, že Newtonova metoda je hodně/málo citlivá na změnu počátečního odhadu a myslíme tíme, že odpovídající matematický problém $\mathcal{P}$ je špatně/dobře podmíněný.


<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Jaký je vztah mezi podmíněností/citlivostí Newtonovy metody a původního problému $\mathbf{F}(\mathbf{x}) = \mathbf{0}$?
- Vysvětlete rozdíl mezi *citlivostí/podmíněností Newtonovy metody* a *stabilitou Newtonovy metody*.
- Co lze na základě grafů v A1 usoudit o citlivosti Newtonovy metody?

</blockquote>
</div>

Vezmeme si funkci $f(x) = x^3 - x$ s kořeny $0$ a $\pm 1$. Následující kód vykresluje sekvenci bodů Newtonovy metody pro tři různé počáteční aproximace $x_0 = 0.4472, x_0 = 1/\sqrt{5} \approx 0.44721\dotsc$ a $x_0 = 0.4473$ (označené indexy "l", "m" a "r" pro *left*, *middle* a *right*).

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Předpokládejme, že v  tomto případě je počítač přesný, tj. obrázky níže odpovídají iteracím Newtonovy metody, které bychom získali výpočtem na papíře. Co lze na základě grafů v A1 usoudit o citlivosti Newtonovy metody?

</blockquote>
</div>

In [ ]:
def f_cubic(x):
  return x**3-x
def df_cubic(x):
  return (3*x**2 - 1)*np.eye(len(x))

maxit = 10
tol = 1e-8

x0 = np.array([0.4472])
(x_approx_l,), _ = newton_method(f_cubic, df_cubic, x0, maxit, tol)

x0 = np.array([1/np.sqrt(5)])
(x_approx_m,), _ = newton_method(f_cubic, df_cubic, x0, maxit, tol)

x0 = np.array([0.4473])
(x_approx_r,), _ = newton_method(f_cubic, df_cubic, x0, maxit, tol)


################################################# Vykreslovani - neni treba menit, pokud nechcete menit experiment
mesh_left = np.linspace(-.5,.5,100)
mesh_mid = np.linspace(-.5,.5,100)
mesh_right = np.linspace(-1,16,100)
plt_opts1 = {'color': 'tab:gray', 'label': r'$f(x)$'}
plt_opts2 = {'linestyle': ':', 'marker': 'o', 'markersize': 7, 'markeredgecolor': 'k', 'label': 'Newton'}
plt.figure(figsize=(16,5))
plt.subplot(1,3,1)
plt.plot(mesh_left, f_cubic(mesh_left), **plt_opts1)
plt.plot(x_approx_l, f_cubic(x_approx_l), color='tab:red', **plt_opts2)
plt.title(r'$x_0 = 0.4472$')
plt.legend()
plt.subplot(1,3,2)
plt.plot(mesh_mid, f_cubic(mesh_mid), **plt_opts1)
plt.plot(x_approx_m, f_cubic(x_approx_m), color='tab:blue', **plt_opts2)
plt.title(r'$x_0 = 1/\sqrt{5}$')
plt.legend()
plt.subplot(1,3,3)
plt.plot(mesh_right, f_cubic(mesh_right), **plt_opts1)
plt.plot(x_approx_r, f_cubic(x_approx_r), color='tab:green', **plt_opts2)
plt.title(r'$x_0 = 0.4473$')
plt.legend()
plt.suptitle("Vyvoj bodu $x_k$ & $f(x_k)$ pro Newtonovu metodu z pocatecni aproximace $x_0$")
plt.show()

### Úloha A3 - Newtonův fraktál

V minulé úloze jsme viděli, že volba počátečního bodu může mít velký dopad na (ne)konvergenci. Zkusíme si s tímto jevem trochu více pohrát. Vezmeme si ***komplexní funkci komlexní proměnné*** $f(z) = z^3-1 \in \mathbb{C}$ s kořeny

\begin{equation*}
z_1 = -\frac{1}{2} + i\frac{\sqrt{3}}{2}, \quad z_2 = -\frac{1}{2} - i\frac{\sqrt{3}}{2}, \quad z_3 = 1.
\end{equation*}

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír & kódění**

- **Tužka a papír**: Přepište $f(z)$ jako funkci $\mathbf{F}(\mathbf{x}) : \mathbb{R}^{2} \rightarrow \mathbb{R}^{2}$ podle klasické identifikace $\mathbb{C}$ s $\mathbb{R}^{2}$, tj.
  - $z\equiv \mathbf{x} = [\operatorname{Re}(z),\operatorname{Im}(z)]$,
  - $\operatorname{Re}(f(z)) = \mathbf{F}(\mathbf{x})_1$,
  - $\operatorname{Im}(f(z)) = \mathbf{F}(\mathbf{x})_2$.

  Pro daný bod $\mathbf{x} = [x_1,x_2]^T \in \mathbb{R}^2$ spočítejte derivaci $J_{\mathbf{x}}\in \mathbb{R}^{2\times 2}$ funkce $\mathbf{F}$.

</blockquote>
</div>

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Naimplementujte funkci `newton_root_index()` podle anotace níže.

</blockquote>
</div>

In [ ]:
def newton_root_index(roots, f, df, x0, max_iter, tol):
    '''
    Pro dany pocatecni bod x0 vratime Newtonova metoda pro reseni soustavy nelinearnich rovnic "f(x)=0".

    Input
    -------
    roots : numpy array rozmeru "PocetKorenu-krat-2". V kazdem radku jsou napsany dve souradnice x1, x2 jednoho z korenu funkce "f"
    -- zbytek inputu je stejny jako u "newton_method()" --

    Output
    -------
    Pokud "newton_method(f,df,x0,maxit,tol)" zkonvergovala k nejakemu z korenu (ve smyslu "||x_k - Koren_1|| < tol"), vratime poradi toho korenu (napr. vratime "1" pokud Newton zkonvergoval k prvnimu korenu, tj. k "roots[0,:]").
    Pokud "newton_method(f,df,x0,maxit,tol)" nezkonvergovala, vratime 0.
    '''
    # Vas kod
    ...

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Využijte funkci `newton_root_index()` k doplnění předpřipraveného kódu v buňce níže a vykreslení tzv. [Newtonova fraktálu](https://en.wikipedia.org/wiki/Newton_fractal). Kód je strukturován do dvou kroků:

  - *Krok 1* : vygenerujeme síť bodů ve čtverci $[-1,1]\times [-1,1]$.
  To uděláme tak, že vygenerujeme rovnoměrně rozložené body $\xi_1,\dotsc ,\xi_N$ v intervalu $[-1,1]$ a pak vezmeme "všechny dvojice" $[\xi_i,\xi_j]$. Naštěstí na tohle existuje funkce [np.meshgrid](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html).

  - *Krok 2* : pro každý bod $[\xi_i,\xi_j] \in \mathbb{R}^2$ otestujeme, ke kterému z kořenů funkce $\mathbf{F}$ konverguje _Newtonova metoda z tohoto bodu pomocí `newton_root_index()`. Výsledek (tj. index $0,1,2$ nebo $3$) si uložíme do pole `basin_sqr1` (a analogicky pro další čtverce).

- Doplňte *Krok 2*: pro každý bod sítě zavolejte `newton_root_index()` a výsledek uložte do `basin_sqr1` / `basin_sqr_half` / `basin_sqr2` pro 3 různé "zoom-in", tj. pro body v $[-0.5,0.5]^2, [-1,1]^2$ a $[-2,2]^2$.
- Obrázky níže odpovídají počtu bodů $N = 350$ (kód poběží cca 2-3 minuty). Zkuste experimentovat s různým počtem bodů.
- Esteticky naladění/é si můžou vybrat v jakých barvách se bude fraktál zobrazovat skrze parametr `colormap` - lze volit z široké nabídky v pythonu, změnou `viridis` na libovolné jiné `keyword` ze [seznamu](https://matplotlib.org/stable/users/explain/colors/colormaps.html).

- [Navíc] Najděte si váš oblíbený Newtonův fraktál různými volbami $f(z)$, např. $f(z) = z^3 - 2z + 2$ nebo $f(z) = \sin(z)$.

>Pokud se chcete dozvědět více o fraktálech, tak sám přednášející má na toto téma výběrovou přednášku [NMNV361: Fraktály a chaotická dynamika](https://www.karlin.mff.cuni.cz/~kucera/Frakt_chaot_dyn.html).

</blockquote>
</div>

In [ ]:
################################################# Vykreslovani - neni treba menit, pokud nechcete menit experiment
def f_complex(x):
  return np.asarray( [x[0]**3 - 3 * x[0] * x[1]**2 - 1, 3 * x[0]**2 * x[1] - x[1]**3] )
def df_complex(x):
  return np.asarray( [[3 * x[0]**2 - 3 * x[1]**2, -6 * x[0] * x[1]], [6 * x[0] * x[1], 3 * x[0]**2 - 3 * x[1]**2]] )

max_iter = 20
tol = 1e-6
N = 350
roots = np.array([ [-1/2, np.sqrt(3)/2] , [-1/2, -np.sqrt(3)/2] , [1,0] ])
colormap = plt.get_cmap('viridis')

### Diskretizace čtverce [-1,1]^2
# Krok 1
mesh_1D = np.linspace(-1, 1, N) # vygenerujeme N bodu xi_1, ..., xi_N na [-1,1] vcetne kraju - napr. "np.linspace(2,3,5) = [2., 2.25, 2.5 , 2.75, 3.]"
X1,X2 = np.meshgrid(mesh_1D, mesh_1D) # body [xi_i,xi_j] ziskame jako [ X1[i,j],X2[i,j] ]
# Krok 2
basin_sqr1 = np.zeros((N, N))
for i in range(0,N):
    for j in range(0,N):
        x0 = np.asarray([X1[i, j], X2[i, j]])
        basin_sqr1[i, j] = newton_root_index(roots, f_complex, df_complex, x0, max_iter, tol)

# Diskretizace čtverce [-0.5,0.5]^2
mesh_1D = np.linspace(-0.5, 0.5, N)
X1,X2 = np.meshgrid(mesh_1D, mesh_1D)
basin_sqr_half = np.zeros((N, N))
for i in range(0,N):
    for j in range(0,N):
        x0 = np.asarray([X1[i, j], X2[i, j]])
        basin_sqr_half[i, j] = newton_root_index(roots, f_complex, df_complex, x0, max_iter, tol)

# Diskretizace čtverce [-2,2]^2
mesh_1D = np.linspace(-2, 2, N)
X1,X2 = np.meshgrid(mesh_1D, mesh_1D)
basin_sqr2 = np.zeros((N, N))
for i in range(0,N):
    for j in range(0,N):
        x0 = np.asarray([X1[i, j], X2[i, j]])
        basin_sqr2[i, j] = newton_root_index(roots, f_complex, df_complex, x0, max_iter, tol)


################################################# Vykreslovani - neni treba menit, pokud nechcete menit experiment
fig = plt.figure(figsize=(16,7))
ax1 = plt.subplot(1,3,1)
plt.imshow(basin_sqr1, extent=[-1, 1, -1, 1], origin='lower', cmap=colormap); plt.xticks([-1, 0, 1]); plt.yticks([-1, 0, 1]); plt.title(r'$\mathbf{x} \in [-1,1]^2$')
ax2 = plt.subplot(1,3,2)
plt.imshow(basin_sqr_half, extent=[-.5, .5, -.5, .5], origin='lower', cmap=colormap); plt.xticks([-.5, 0, .5]); plt.yticks([-.5, 0, .5]); plt.title(r'$\mathbf{x} \in [-0.5,0.5]^2$')
ax3 = plt.subplot(1,3,3)
plt.imshow(basin_sqr2, extent=[-2, 2, -2, 2], origin='lower', cmap=colormap); plt.xticks([-2, 0, 2]); plt.yticks([-2, 0, 2]); plt.title(r'$\mathbf{x} \in [-2,2]^2$')
cbar = plt.colorbar(ticks = [0, 1, 2, 3], location = 'bottom', ax =np.asarray([ax1,ax2,ax3]), shrink=.4, aspect=40, pad=0.085)
plt.show()

### Úloha A4

Newtonova metoda je sice primárně metoda určená pro řešení *nelineárních* algebraických rovnic, ale může být zjevně (nic nám v tom nebrání) i pro řešení problémů z první části semestru. Vezmeme si matici $A\in \mathbb{R}^{2\times 2}$ a $\mathbf{b} \in \mathbb{R}^2$, a chceme najít vektor $\mathbf{x} \in \mathbf{R}^2$ splňující
$A \mathbf{x}= \mathbf{b}$ , neboli $
\begin{bmatrix}
a_{11} & a_{12} \\ a_{21} & a_{22}
\end{bmatrix}
\begin{bmatrix}
x_1 \\ x_2
\end{bmatrix} =
\begin{bmatrix}
b_1 \\ b_2
\end{bmatrix}$

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**


- Zadefinujte odpovídající funkci $\mathbf{F}(\mathbf{x}) \, : \, \mathbb{R}^2 \rightarrow \mathbb{R}^2$ pro kterou je $\mathbf{x}$ kořenem.

- Odvoďte explicitní předpis Newtonovy metody pro dané $A$ a $\mathbf{b}$.

- Okomentujte praktičnost/použitelnost Newtonovy metody pro soustavy lineárních rovnic v praxi.

</blockquote>
</div>



### ZÚ - analýza konvergence Newtonovy metody

Na základě A3 by se mohlo zdát, že ikdyž je teorie "pesimistická", tak Newton vlastně "skoro vždycky konverguje" a jen potřebujeme lepší teorii/popis, abychom to byli schopni dokázat. Pro některé třídy funkcí tomu tak opravdu je.

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

Vezmeme si hladkou, rostoucí a ryze konvexní funkci $f : \mathbb{R}\rightarrow \mathbb{R}$ s kořenem $x^*$. Ukažte, že Newtonova metoda bude konvergovat pro libovolné $x_0$.
  - Co by se změnilo, kdyby $f$ byla ryze konkávní a/nebo klesající?
  - Lze zeslabit dané vlastnosti $f$ z $\mathbb{R}$ na jiný, menší interval?

- [Nápověda & hinty](#head-w08-hinty-zu-analyza-newtonovy-metody)

</blockquote>
</div>

### ZÚ - analýza citlivosti Newtonovy metody

Na základě A3 (a předchozí úlohy) by se mohlo zdát, že ikdyž je teorie "pesimistická", tak Newton vlastně "skoro vždycky konverguje". Realita je bohužel jiná a to i "pouze" v jedné proměnné, natož pro vektorové funkce. Zobecníme funkce z A1 a vezmeme si

\begin{equation*}
f_{\lambda}(x) = x^3 - x + \lambda,
\end{equation*}

funkci, která je parametrizovaná skalárem $\lambda$, který může být reálný nebo komplexní. V A1 jsme viděli, že metoda se pro $x_0=1/\sqrt{5}$ zasekla v tzv. ***2-cyklu*** a oscilovala mezi dvěma body a pro jiné $x_0$ naopak konvergovala k jednomu z kořenů.

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

- Napište explicitní vzorec pro jeden krok Newtonovy metody, tj. najděte funkci $g_{\lambda}(x)$ takovou, že platí

  \begin{equation*}
  x_{k+1} = g_{\lambda}(x_k).
  \end{equation*}

- Odvoďte pro které $x_k$ a $\lambda$ bude Newtonova metoda  oscilovat ve 2-cyklu, tj. bude platit, že pro všechna $\ell = 0,1,2,\dotsc$ platí

  \begin{equation*}
  x_{k+2\ell} = x_{k} \quad \text{a} \quad x_{k+2\ell + 1} = x_{k+1}.
  \end{equation*}

  Není zapotřebí tuto podmínku vyřešit, stačí ji nechat v implicitní formě, např.
  \begin{equation*} \text{musi platit:} \quad \sin(2\pi \lambda) + e^{x_k} = \sqrt{2}\end{equation*}

  - Jak by vypadala analogická podmínka pro 3-cyklus nebo 17-cyklus?
  - Jak by vypadala podmínka pro $x_k$ a $\lambda$, aby Newtonova metoda zkonvergovala po 3 nebo 17 iterací?
  - Jak by vypadala podmínka pro $x_k$ a $\lambda$, aby Newtonova metoda  "crashnula" (tj. nulová derivace $f$ v bodě $x_k$) po 3 nebo 17 iterací?
  - Jak tyto podmínky závisí na $x_k$?

- Vzpoměňte si na poslední část [cvičení o podmíněnosti a stabilitě](week-01-conditioning-stability) v kontextu pozorování výše.
  - Co lze/nelze říci o citlivosti Newtonovy metody pro polynomiální funkce?
  - Co lze/nelze říci o stabilitě Newtonovy metody pro polynomiální funkce?
  - Okomentujte vaše závěry v kontextu výsledků o citlivosti kořenů Wilkinsonova polynomu.

</blockquote>
</div>

## Část B: Fixed-point metody

Pro danou funkci $\mathbf{F}$ se snažíme najít novou funkci $\mathbf{G}$ takovou, že

  - kořeny $\mathbf{F}$ jsou pevné body $\mathbf{G}$
  - na $\mathbf{G}$ lze aplikovat Banachova věta o pevném bodu/kontrakci.

Pro Newtonovu metodu jsme viděli (na přednášce i v A1), že konvergence je asymptoticky kvadratická, zatímco podle Banachovy věty o kontrakci je pro fixed-point metody zaručena "pouze" lineární rychlost konvergence.

### Úloha B1 - lineární vs. kvadratická konvergence

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Newtonova metoda za určitých předpokladů konverguje ***kvadraticky***, tedy platí
  \begin{equation*}\lim\limits_{k\rightarrow +\infty} \frac{ |x^{*}-x_{k+1}| }{ |x^{*}-x_k|^2 }=C,\end{equation*}
  pro nějakou konstantu $C>0$. Předpokládejme, že máme konkrétní problém, kde  $|x^{*}-x_0|=0.1$ a pro který Newtonova metoda splňuje
  \begin{equation*} |x^{*}-x_{k+1}| = |x^{*}-x_k|^2\end{equation*}
  a jistě je tedy splněna i definice kvadratické konvergence s $C\equiv 1$.
    - Kolik potřebujeme iterací na to, abychom chybu zmenšili o řád? A o 7 řádů?

- Fixed-point metody konvergují dle Banachovy věty o pevném bodě alespoň ***lineárně***, tedy platí
  \begin{equation*}\lim\limits_{k\rightarrow +\infty} \frac{ |x^{*}-x_{k+1}| }{ |x^{*}-x_k| }=q,\end{equation*}
  pro nějakou konstantu $q \in (0,1)$. Předpokládejme, že máme konkrétní problém, kde $|x^{*}-x_0|=0.1$ a pro který naše fixed-point metoda splňuje
  \begin{equation*} |x^{*}-x_{k+1}| = \frac{1}{10} |x^{*}-x_k| \end{equation*}
  a jistě je tedy splněna i definice lineární konvergence s $q=1/10=C/10$.
    - Kolik potřebujeme iterací na to, abychom chybu zmenšili o řád? A o 7 řádů?

</blockquote>
</div>

### Úloha B2 - odvození fixed-point metod

Začneme opět v jedné dimenzi a to s funkcí

\begin{equation*}
f(x) = x^2 -x - 2
\end{equation*}

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Odvoďte funkce

  \begin{equation*}
  g_1(x) =x^2-2, \quad g_2(x) =\sqrt{x+2},\\
  g_3(x) =1+\dfrac{2}{x}, \quad g_4(x) = \dfrac{x^2+2}{2x-1}.\\
  \end{equation*}

  jako fixed-point metody pro $f(x)$, tj. ukažte jestli všechny nebo některé kořeny $f$ jsou pevné body funkcí $g_i(x), i=1\dotsc ,4$.

  - Vyberte si jednu z funkcí $g_i(x), i=1,\dotsc ,4$ a odvoďte zda/na jaké množině lze použít Banachovu větu o pevném bodě pro $g_i$ a tím numericky aproximovat kořen/kořeny $f$.

  - Odvoďte iterační předpis Newtonovy metody pro $f$.

</blockquote>
</div>

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Pro $g_1,\dotsc,g_4$ doplňte cyklus, který spočítá vývoj chyby $|{-1}-x_k|$ a $|2-x_k|$ pro vhodné počáteční body.

</blockquote>
</div>

In [ ]:
def g1(x):
    return x**2-2
def g2(x):
    return np.sqrt(x+2)
def g3(x):
    return 1+2/x
def g4(x):
    return (x**2 +2)/(2*x-1)


# Vas kod
N = 20

### g1
...

### g2
...

### g3
...

### g4
...


################################################# Vykreslovani - neni treba menit, pokud nechcete menit experiment
from matplotlib.ticker import MaxNLocator

plt.figure(figsize=(25,5))
ax1 = plt.subplot(1,4,1)
plt.semilogy(err_g1_root1, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k', label='$|-1 - x_k|$');
plt.semilogy(err_g1_root2, 'o:', ms = 7, color = 'tab:blue', markeredgecolor='k', label='$|2 - x_k|$'); plt.title(r'$g_1(x) = x^2-2$'); plt.xlabel("iterace $k$"); plt.legend()
ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

ax2 = plt.subplot(1,4,2, sharey=ax1)
plt.semilogy(err_g2_root1, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k', label='$|-1 - x_k|$');
plt.semilogy(err_g2_root2, 'o:', ms = 7, color = 'tab:blue', markeredgecolor='k', label='$|2 - x_k|$'); plt.title(r'$g_2(x) = \sqrt{x+2}$'); plt.xlabel("iterace $k$"); plt.legend()
ax2.xaxis.set_major_locator(MaxNLocator(integer=True))

ax3 = plt.subplot(1,4,3, sharey=ax1)
plt.semilogy(err_g3_root1, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k', label='$|-1 - x_k|$');
plt.semilogy(err_g3_root2, 'o:', ms = 7, color = 'tab:blue', markeredgecolor='k', label='$|2 - x_k|$'); plt.title(r'$g_3(x) = 1+\frac{2}{x}$'); plt.xlabel("iterace $k$"); plt.legend()
ax3.xaxis.set_major_locator(MaxNLocator(integer=True))

ax4 = plt.subplot(1,4,4, sharey=ax1)
plt.semilogy(err_g4_root1, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k', label='$|-1 - x_k|$');
plt.semilogy(err_g4_root2, 'o:', ms = 7, color = 'tab:blue', markeredgecolor='k', label='$|2 - x_k|$'); plt.title(r'$g_4(x) = \frac{x^2+2}{2x-1}$'); plt.xlabel("iterace $k$"); plt.legend()
ax4.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

### Úloha B3 - fixed-point chaos

Nyní si vezmeme další parametrizovanou funkci, tentokrát

  \begin{equation*}
  f_{\lambda}(x) = -\lambda x^2 + (\lambda-1)x,
  \end{equation*}

pro $\lambda \in [0,4]$ a $x\in [0,1]$.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Odvoďte fixed-point metodu
  \begin{equation*} g_{\lambda}(x) = \lambda x(1-x), \end{equation*}
  opět parametrizovanou reálným skalárem $\lambda \in [0,4]$.

- Odvoďte zda jsou kořeny $f_{\lambda}$ pevné body $g_{\lambda}$.

- Pro které $\lambda \in [0,4]$ lze použít Banachovu větu o kontrakci pro $g_{\lambda}$ na nějaké podmnožině $[0,1]$?

</blockquote>
</div>

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Pro každé $\lambda \in \{1, 2.8, 3.1, 3.5, 3.9\}$ doplňte cyklus, který z počátečního bodu $x_0=0.2$ vygeneruje posloupnost aproximací `xcoords_l*` & `ycoords_l*` a chybu `err_l*` vůči pevnému bodu $g_{\lambda}$, ke kterému `xcoords_l*` konvergují (pokud nekonvergují, vyberte si libovolný).

</blockquote>
</div>

In [ ]:
def plot_cobweb(x0, fx, axis):
    '''Funkce pouzivana k vykreslovani tzv. cobweb grafu v druhem radku obrazku nize.'''
    x_coords = np.nan * np.zeros(2 * len(fx))
    y_coords = np.nan * np.zeros_like(x_coords)
    x_coords[0] = x0
    x_coords[1] = x0
    y_coords[0] = 0
    y_coords[1] = fx[0]
    for i in range(0, len(fx) - 1):
        x_coords[2 + 2*i] = fx[i]
        x_coords[2 + 2*i + 1] = fx[i]
        y_coords[2 + 2*i] = x_coords[2 + 2*i]
        y_coords[2 + 2*i + 1] = fx[i + 1]
    colors = ['tab:blue', 'tab:green', 'tab:orange', 'tab:brown']
    colorind = 0
    for ind in range(0, len(x_coords), 2):
        axis.plot(x_coords[ind:ind+2], y_coords[ind:ind+2], 'o--', ms=5, color=colors[colorind])
        axis.plot(x_coords[ind+1:ind+3], y_coords[ind+1:ind+3], 'o:', ms=5, color=colors[colorind])
        colorind = (colorind + 1) % len(colors)
    return
######################################################################################

def g(x,lambd):
  return lambd * x * (1-x)

N = 300;

### lambda = 1
x0 = 0.2
lambda1 = 1
fixedpnt = 0
xcoords_l1 = np.nan*np.zeros(N)
ycoords_l1 = np.nan*np.zeros(N)
err_l1 = np.nan*np.zeros(N)
...

### lambda = 2.8
x0 = 0.2
lambda2 = 2.8
fixedpnt = (lambda2-1)/lambda2
xcoords_l2 = np.nan*np.zeros(N)
ycoords_l2 = np.nan*np.zeros(N)
err_l2 = np.nan*np.zeros(N)
...

### lambda = 3.1
x0 = 0.2
lambda3 = 3.1
fixedpnt = (lambda3-1)/lambda3
xcoords_l3 = np.nan*np.zeros(N)
ycoords_l3 = np.nan*np.zeros(N)
err_l3 = np.nan*np.zeros(N)
...

### lambda = 3.5
x0 = 0.2
lambda4 = 3.5
fixedpnt = (lambda4-1)/lambda4
xcoords_l4 = np.nan*np.zeros(N)
ycoords_l4 = np.nan*np.zeros(N)
err_l4 = np.nan*np.zeros(N)
...

### lambda = 3.9
x0 = 0.2
lambda5 = 3.9
fixedpnt = (lambda5-1)/lambda5
xcoords_l5 = np.nan*np.zeros(N)
ycoords_l5 = np.nan*np.zeros(N)
err_l5 = np.nan*np.zeros(N)
...

################################################# Vykreslovani
plotmesh = np.linspace(0,1,1000);
plt.figure(figsize=(25,11))

plt.subplot(3,5,1)
plt.plot(xcoords_l1,ycoords_l1,'ro:')
plt.plot(plotmesh,plotmesh,'--',color='tab:gray'); plt.plot(plotmesh,g(plotmesh,lambda1),'k-'); plt.title(r'$\lambda = 1$'); plt.ylabel("$y$"); plt.xlabel("$x$");
ax = plt.subplot(3,5,6)
plotmesh_zoom = np.linspace(0,0.2,1000);
plot_cobweb(0.2,ycoords_l1,ax);  plt.ylabel("$y$"); plt.xlabel("$x$");
plt.plot(plotmesh_zoom,plotmesh_zoom,'--',color='tab:gray'); plt.plot(plotmesh_zoom,g(plotmesh_zoom,lambda1),'k-');
plt.subplot(3,5,11)
plt.xlabel("iterace $k$"); plt.ylabel("$|x^* - x_k|$"); plt.semilogy(err_l1, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k', label='konv. k 0');

plt.subplot(3,5,2)
plt.plot(xcoords_l2,ycoords_l2,'ro:'); plt.plot(plotmesh,plotmesh,'--',color='tab:gray', label='y=x'); plt.plot(plotmesh,g(plotmesh,lambda2),'k-'); plt.title(r'$\lambda = 2.8$'); plt.xlabel("$x$");
ax = plt.subplot(3,5,7)
plotmesh_zoom = np.linspace(0.15,0.7,1000);
plot_cobweb(0.2,ycoords_l2,ax); plt.xlabel("$x$");
plt.plot(plotmesh_zoom,plotmesh_zoom,'--',color='tab:gray'); plt.plot(plotmesh_zoom,g(plotmesh_zoom,lambda2),'k-');
plt.subplot(3,5,12)
plt.xlabel("iterace $k$"); plt.semilogy(err_l2, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k');

plt.subplot(3,5,3)
plt.plot(xcoords_l3,ycoords_l3,'ro:'); plt.plot(plotmesh,plotmesh,'--',color='tab:gray'); plt.plot(plotmesh,g(plotmesh,lambda3),'k-'); plt.title(r'$\lambda = 3.1$'); plt.xlabel("$x$");
ax = plt.subplot(3,5,8)
plotmesh_zoom = np.linspace(0.15,0.8,1000);
plot_cobweb(0.2,ycoords_l3,ax); plt.xlabel("$x$");
plt.plot(plotmesh_zoom,plotmesh_zoom,'--',color='tab:gray'); plt.plot(plotmesh_zoom,g(plotmesh_zoom,lambda3),'k-');
plt.subplot(3,5,13)
plt.xlabel("iterace $k$"); plt.semilogy(err_l3, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k');

plt.subplot(3,5,4)
plt.plot(xcoords_l4,ycoords_l4,'ro:'); plt.plot(plotmesh,plotmesh,'--',color='tab:gray'); plt.plot(plotmesh,g(plotmesh,lambda4),'k-'); plt.title(r'$\lambda = 3.5$'); plt.xlabel("$x$");
ax = plt.subplot(3,5,9)
plotmesh_zoom = np.linspace(0.15,0.9,1000);
plot_cobweb(0.2,ycoords_l4,ax); plt.xlabel("$x$");
plt.plot(plotmesh_zoom,plotmesh_zoom,'--',color='tab:gray'); plt.plot(plotmesh_zoom,g(plotmesh_zoom,lambda4),'k-');
plt.subplot(3,5,14)
plt.xlabel("iterace $k$"); plt.semilogy(err_l4, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k');

plt.subplot(3,5,5)
plt.plot(xcoords_l5,ycoords_l5,'ro:'); plt.plot(plotmesh,plotmesh,'--',color='tab:gray'); plt.plot(plotmesh,g(plotmesh,lambda5),'k-'); plt.title(r'$\lambda = 3.9$'); plt.xlabel("$x$");
ax = plt.subplot(3,5,10)
plotmesh_zoom = np.linspace(0.15,0.95,1000);
plot_cobweb(0.2,ycoords_l5,ax); plt.xlabel("$x$");
plt.plot(plotmesh_zoom,plotmesh_zoom,'--',color='tab:gray'); plt.plot(plotmesh_zoom,g(plotmesh_zoom,lambda5),'k-');
plt.subplot(3,5,15)
plt.xlabel("iterace $k$"); plt.semilogy(err_l5, 'o:', ms = 7, color = 'tab:red', markeredgecolor='k');
################################################# Vykreslovani

### ZÚ - Fixed-point metody jako staří známí

Vezmeme si čtvercovou matici $A\in \mathbb{R}^{n\times n}$ a vektor $\mathbf{b}\in \mathbb{R}^{n}$.

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

Odvoďte stacionární metodu pro $A\mathbf{x}=\mathbf{b}$ založenou na štěpení $A=M-N$ jako fixed-point metodu.
  - Odvoďte zobrazení $G:\mathbb{R}^{n} \rightarrow \mathbb{R}^{n}$, na které byste Banachovu větu použili.
  - Odvoďte, za jakých podmínek na normu matice tohoto zobrazení $G$ lze Banachovu větu použít.
  - Porovnejte s podmínkami konvergence ze [cvičení o iteračních metodách](week-02-iterative-methods).

</blockquote>
</div>



Ve [cvičení o iteračních metodách](week-02-iterative-methods) jsme viděli, že nutnou a postačující podmínkou konvergence je $\rho(M^{-1}N)<1$. Snadno nahlédneme, že podmínka z předchozího úkolu tuto nerovnost implikuje.

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

Zkuste nalézt příklad lineárního zobrazení $L(\mathbf{x}) = B\mathbf{x}$ s maticí $B$ takovou, že $\rho(B)<1$, které však není kontraktivní.

</blockquote>
</div>

Splnění výše uvedené podmínky nám tedy ještě neumožňuje aplikovat standardní Banachovu větu přímo. Místo toho lze použít následující "iterovaná" verze:
 - Uvažujme zobrazení $F: \mathbb{R}^{n} \rightarrow \mathbb{R}^{n}$, pro které existuje $\tilde{k}\in\mathbb{N}$ tak, že $F^{\tilde{k}}$ je kontrakce. Potom $F$ má právě jeden pevný bod.

 Důkaz tohoto tvrzení se opírá o standardní Banachovu větu a je velice snadný, proto jej necháme jako dobrovolné cvičení. Případně jej lze nalézt ve skriptech z matematické analýzy [Pick et al., Příklad 8.10.38. na str. 626 ve verzi ze dne 7.6.2026](https://www.karlin.mff.cuni.cz/~pick/analyza%20pro%20studenty%202026-06-07.pdf).

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

- Předpokládejme, že $\rho(M^{-1}N)<1$. Dokažte, že existuje $\tilde{k}\in\mathbb{N}$, pro které je $G^{\tilde{k}}$ kontrakce. Co nám potom říká "iterovaná" Banachova věta?
- [Nápovědy & hinty](#head-w08-hinty-zu-fixed-point-metody)

</blockquote>
</div>

### Navíc na zamyšlení (není potřeba k získání zápočtu)

Pokud bychom opravdu toužili po kontraktivním zobrazení (což se může v některých aplikacích stát), nezbývá než změnit normu prostoru, ve kterém pracujeme (je dobré si připomenout, že na vektorových prostorech konečné dimenze jsou všechny normy ekvivalentní). Ke konstrukci této nové normy budeme potřebovat následující velice užitečnou větu (tzv. Gelfandův vzorec)

- Nechť $B\in\mathbb{C}^{n\times n}$, potom
\begin{equation*}
\rho(B)=\lim_{k\rightarrow ∞}\|B^k\|^{\frac{1}{k}}.
\end{equation*}

Ještě se nám bude hodit Cauchyovo odmocninové kritérium v následující podobě

- Nechť $\lbrace a_k \rbrace^{∞}_{k=0}$ je posloupnost nezáporných reálných čísel splňující podmínku
\begin{equation*}
\lim_{k→∞}a^{\frac{1}{k}}_{k}<1,
\end{equation*}
potom řada $\sum^{\infty}_{k=0}a_k$ konverguje.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

- Nechť je dána matice $B\in\mathbb{C}^{n\times n}$ taková, že $\rho(B)<1$. Nalezněte explicitní vzorec pro normu $N_B$ na $\mathbb{C}^n$ takovou, že existuje $α\in(0,1)$ splňující
 \begin{equation*}
 N_B(B\mathbf{x})\leq αN_B(\mathbf{x})
 \end{equation*}
pro každé $\mathbf{x}\in\mathbb{C}^n$. Jinými slovy, lineární zobrazení s maticí $B$ je kontrakce v normovaném lineárním prostoru $(\mathbb{C}^n,N_B(\cdot))$.
- [Nápověda & hinty](#head-w08-hinty-zu-fixed-point-metody)

</blockquote>
</div>

## Hinty

<a name="head-w08-hinty-zu-analyza-newtonovy-metody"></a>
### Hinty ZU - analyza Newtonovy metody

#### Věta o konvergenci monotónní posloupnosti

Zkuste použít [Větu o konvergenci omezené neklesající/nerostoucí posloupnosti](https://en.wikipedia.org/wiki/Monotone_convergence_theorem) :

> Mějme posloupnosti $a_1 \leq a_2 \leq a_3 \leq \dotsc \leq M \in \mathbb{R}$ a $b_1 \geq b_2 \geq b_3 \geq \dotsc \geq m \in \mathbb{R}$. Obě tyto posloupnosti konvergují a to k $\sup a_i$ a $\inf b_i$.

Co tedy pro posloupnost generovanou Newtonovou metodou musím ukázat?

#### Nerovnost pro konvexní/konkávní funkce

Může se hodit, že pro konvexní diferencovatelnou funkci $f$ platí nerovnost tvaru:
\begin{equation*}
 \forall x,y \, : \, f(x) \geq f(y) + f'(y)(x-y)
\end{equation*}

Co platí pro konkávní funkce?

<a name="head-w08-hinty-zu-fixed-point-metody"></a>
### Hinty ZU - Fixed-point metody

#### Limita $B^k$ a spektrální poloměr

Tady se může hodit věta z přednášky: Nechť $B\in\mathbb{C}^{n\times n}$, potom
\begin{equation*}
\lim_{k\rightarrow \infty} B^k = 0 \quad \iff \quad \rho(B)<1.
\end{equation*}
Co se potom dá říci o limitě $\|B^k\|$?

#### Konstrukce normy $N_B$ (bonus)

Hledejme normu ve tvaru řady
\begin{equation*}
N_B(\mathbf{x})=\sum_{k=0}^∞\frac{a_k(\mathbf{x})}{\eta^k}
\end{equation*}
pro vhodné kladné reálné číslo $\eta$. Toto číslo a posloupnost $\lbrace a_k(\mathbf{x})\rbrace$ chceme volit tak, aby se pro důkaz konvergence řady dalo použít Cauchyovo odmocninové kritérium a Gelfandův vzorec. Jaké axiomy musí splňovat norma?

#### Kontraktivita posunutím indexů (bonus)

Kontraktivitu zkuste ukázat pomocí vhodného posunutí indexů v sumě.